# Extracción de datos farmacológicos mediante la API de openFDA

## 1. Objetivo

El objetivo de este proyecto es desarrollar un pipeline en Python capaz de conectarse a una API pública, realizar peticiones GET, recuperar información farmacológica y almacenarla en un formato estructurado.

Como fuente de datos se utiliza la API de **openFDA Drug Labeling**, que proporciona información proveniente de etiquetas oficiales de medicamentos. La selección de esta API permite trabajar con datos relacionados con farmacología, incluyendo información sobre farmacocinética, farmacodinamia y farmacogenómica.

El proyecto se desarrolla aplicando buenas prácticas básicas de seguridad, evitando almacenar credenciales directamente en el código y utilizando variables de entorno para gestionar la API Key.

### Objetivos específicos

- Conectarse a la API pública de openFDA mediante peticiones HTTP GET.
- Gestionar la API Key mediante variables de entorno.
- Explorar y seleccionar campos farmacológicos relevantes.
- Realizar una transformación básica de la respuesta obtenida.
- Almacenar los datos en un archivo estructurado.
- Incorporar manejo de errores y validación de las respuestas HTTP.
- Dejar el proceso documentado y reproducible.

## 2. Fuente de datos: openFDA

**openFDA** es una iniciativa de la U.S. Food and Drug Administration (FDA) que proporciona acceso programático a determinados conjuntos de datos regulatorios y de salud pública mediante APIs.

Para este proyecto se utilizará el endpoint **Drug Labeling**, que permite consultar información contenida en etiquetas de medicamentos.

La consulta se realizará mediante peticiones HTTP GET al siguiente endpoint:

`https://api.fda.gov/drug/label.json`

Entre los campos disponibles se encuentran, entre otros:

- `generic_name`
- `brand_name`
- `active_ingredient`
- `route`
- `mechanism_of_action`
- `pharmacokinetics`
- `pharmacodynamics`
- `pharmacogenomics`
- `clinical_pharmacology`
- `indications_and_usage`
- `adverse_reactions`
- `drug_interactions`

Para esta primera etapa se priorizarán los campos relacionados con la farmacología del medicamento. Los textos farmacológicos se conservarán sin aplicar técnicas de procesamiento de lenguaje natural, ya que el objetivo de esta actividad es la extracción y almacenamiento seguro de datos mediante una API.

## 3. Configuración del entorno

El proyecto utiliza un entorno independiente de Python denominado `openfda_ds`.

Las principales librerías utilizadas son:

- `requests`: realizar peticiones HTTP a la API.
- `python-dotenv`: cargar variables de entorno desde el archivo `.env`.
- `pandas`: facilitar la transformación y organización de los datos.
- `pyarrow`: permitir el almacenamiento posterior en formato Parquet.

La API Key se almacena localmente en un archivo `.env` y no se incluye directamente en el código fuente. Este archivo será excluido del repositorio mediante `.gitignore`.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")

print("API Key cargada:", API_KEY is not None)

API Key cargada: True


## 4. Prueba de conexión con la API

Antes de realizar una extracción de mayor volumen, se realiza una consulta pequeña al endpoint de Drug Labeling para comprobar que:

- la API Key se encuentra disponible;
- el endpoint responde correctamente;
- la petición utiliza el método HTTP `GET`;
- la respuesta contiene información en formato JSON.

En esta etapa se solicitarán solamente algunos registros como prueba. La extracción definitiva se realizará posteriormente mediante un script independiente.

In [4]:
BASE_URL = "https://api.fda.gov/drug/label.json"

params = {
    "api_key": API_KEY,
    "search": "pharmacokinetics:[* TO *]",
    "limit": 5
}

try:
    response = requests.get(BASE_URL, params=params, timeout=30)

    if response.status_code == 200:
        data = response.json()
        print("Conexión exitosa con openFDA.")
        print(f"Registros recuperados: {len(data.get('results', []))}")
    else:
        print(f"Error HTTP {response.status_code}: {response.text[:300]}")

except requests.exceptions.RequestException as e:
    print(f"Error al realizar la petición: {e}")

Conexión exitosa con openFDA.
Registros recuperados: 5


## 5. Exploración de la respuesta de la API

La respuesta de openFDA se encuentra en formato JSON y contiene diferentes elementos. El objeto `results` almacena los registros recuperados mediante la consulta.

En esta etapa se inspeccionará la estructura de los registros para identificar los campos disponibles y seleccionar aquellos que resulten relevantes para el objetivo del proyecto.

La exploración se limita inicialmente a los registros obtenidos en la prueba de conexión, antes de realizar la extracción definitiva.

In [6]:
# Cantidad de elementos principales de la respuesta
print(data.keys())

dict_keys(['meta', 'results'])


In [7]:
# Inspeccionamos las claves disponibles en el primer registro
first_record = data["results"][0]

print(first_record.keys())

dict_keys(['spl_product_data_elements', 'spl_unclassified_section', 'description', 'clinical_pharmacology', 'clinical_pharmacology_table', 'pharmacokinetics', 'microbiology', 'microbiology_table', 'clinical_studies', 'indications_and_usage', 'spl_unclassified_section_table', 'contraindications', 'warnings', 'precautions', 'general_precautions', 'information_for_patients', 'drug_interactions', 'carcinogenesis_and_mutagenesis_and_impairment_of_fertility', 'pregnancy', 'nonteratogenic_effects', 'nursing_mothers', 'pediatric_use', 'geriatric_use', 'adverse_reactions', 'dosage_and_administration', 'dosage_and_administration_table', 'how_supplied', 'package_label_principal_display_panel', 'set_id', 'id', 'effective_time', 'version', 'openfda'])


In [8]:
print("Nombre genérico:")
print(first_record.get("openfda", {}).get("generic_name"))

print("\nPrincipio activo:")
print(first_record.get("active_ingredient"))

print("\nFarmacocinética:")
print(first_record.get("pharmacokinetics"))

print("\nFarmacodinamia:")
print(first_record.get("pharmacodynamics"))

print("\nMecanismo de acción:")
print(first_record.get("mechanism_of_action"))

Nombre genérico:
None

Principio activo:
None

Farmacocinética:
['Pharmacokinetics: Serum, urine and tear concentrations of ofloxacin were measured in 30 healthy women at various time points during a ten-day course of treatment with ofloxacin ophthalmic solution. The mean serum ofloxacin concentration ranged from 0.4 ng/mL to 1.9 ng/mL. Maximum ofloxacin concentration increased from 1.1 ng/mL on day one to 1.9 ng/mL on day 11 after QID dosing for 10 1/2 days. Maximum serum ofloxacin concentrations after ten days of topical ophthalmic dosing were more than 1000 times lower than those reported after standard oral doses of ofloxacin. Tear ofloxacin concentrations ranged from 5.7 to 31 mcg/g during the 40 minute period following the last dose on day 11. Mean tear concentration measured four hours after topical ophthalmic dosing was 9.2 mcg/g. Corneal tissue concentrations of 4.4 mcg/mL were observed four hours after beginning topical ocular application of two drops of ofloxacin ophthalmic 

## 6. Exploración de los campos de identificación

Los registros de Drug Labeling contienen información de identificación y clasificación del producto dentro del objeto `openfda`.

Estos campos serán explorados para determinar cuáles pueden incorporarse al dataset final junto con la información farmacológica y clínica.

In [9]:
print(first_record["openfda"].keys())

dict_keys([])


In [11]:
openfda_fields = first_record.get("openfda", {})

for field, value in openfda_fields.items():
    print(f"{field}: {value}")

### Disponibilidad de campos

La estructura de los registros de Drug Labeling puede variar entre productos. Por este motivo, antes de definir el esquema final del dataset se analiza la disponibilidad de los principales campos en el conjunto de registros obtenido durante la prueba.

Esta evaluación permite seleccionar variables relevantes sin asumir que todos los campos están presentes en todos los registros.

In [12]:
# Campos que queremos evaluar
fields_to_check = [
    "description",
    "clinical_pharmacology",
    "pharmacokinetics",
    "pharmacodynamics",
    "pharmacogenomics",
    "mechanism_of_action",
    "indications_and_usage",
    "contraindications",
    "warnings",
    "precautions",
    "adverse_reactions",
    "drug_interactions",
    "dosage_and_administration",
    "clinical_studies"
]

# Analizamos cuántos de los registros contienen cada campo
field_availability = {}

for field in fields_to_check:
    count = sum(
        1 for record in data["results"]
        if field in record and record[field]
    )
    field_availability[field] = count

for field, count in field_availability.items():
    print(f"{field}: {count}/{len(data['results'])}")

description: 5/5
clinical_pharmacology: 5/5
pharmacokinetics: 5/5
pharmacodynamics: 3/5
pharmacogenomics: 0/5
mechanism_of_action: 3/5
indications_and_usage: 5/5
contraindications: 5/5
warnings: 2/5
precautions: 2/5
adverse_reactions: 5/5
drug_interactions: 5/5
dosage_and_administration: 5/5
clinical_studies: 4/5


## 7. Definición de las variables a extraer

A partir de la exploración inicial de los registros se definió un conjunto de variables organizado en tres categorías:

### Identificación y características del producto

- `brand_name`
- `generic_name`
- `manufacturer_name`
- `product_type`
- `route`
- `dosage_form`
- `substance_name`
- `product_ndc`

### Información farmacológica

- `clinical_pharmacology`
- `pharmacokinetics`
- `pharmacodynamics`
- `pharmacogenomics`
- `mechanism_of_action`

### Información clínica y de seguridad

- `indications_and_usage`
- `contraindications`
- `warnings`
- `precautions`
- `adverse_reactions`
- `drug_interactions`
- `dosage_and_administration`
- `clinical_studies`

Además, se conservarán identificadores y metadatos del registro:

- `set_id`
- `id`
- `effective_time`

La selección busca mantener información relevante para futuros análisis de farmacología y Data Science, incluyendo posibles aplicaciones posteriores de Machine Learning. 

Debido a que la estructura de los registros puede variar, algunos campos pueden presentar valores ausentes. La disponibilidad observada durante la exploración inicial no se considera representativa de todo el dataset y será evaluada nuevamente sobre la extracción completa.

## 8. Transformación básica

La respuesta de openFDA contiene campos distribuidos en distintos niveles del objeto JSON. En esta etapa se seleccionan y organizan únicamente las variables definidas previamente, separando los campos generales de los contenidos dentro de `openfda`.

Los campos que no estén disponibles en un registro se conservarán como valores nulos, evitando descartar registros completos por la ausencia de una variable.

In [13]:
def get_first_value(dictionary, field):
    """Obtiene el primer valor de un campo de openFDA."""
    value = dictionary.get(field)
    
    if isinstance(value, list):
        return value[0] if value else None
    
    return value


def transform_record(record):
    """Transforma un registro de openFDA en una estructura plana."""
    
    openfda = record.get("openfda", {})
    
    transformed = {
        # Identificación del producto
        "brand_name": get_first_value(openfda, "brand_name"),
        "generic_name": get_first_value(openfda, "generic_name"),
        "manufacturer_name": get_first_value(openfda, "manufacturer_name"),
        "product_type": get_first_value(openfda, "product_type"),
        "route": get_first_value(openfda, "route"),
        "dosage_form": get_first_value(openfda, "dosage_form"),
        "substance_name": get_first_value(openfda, "substance_name"),
        "product_ndc": get_first_value(openfda, "product_ndc"),
        
        # Información farmacológica
        "clinical_pharmacology": get_first_value(record, "clinical_pharmacology"),
        "pharmacokinetics": get_first_value(record, "pharmacokinetics"),
        "pharmacodynamics": get_first_value(record, "pharmacodynamics"),
        "pharmacogenomics": get_first_value(record, "pharmacogenomics"),
        "mechanism_of_action": get_first_value(record, "mechanism_of_action"),
        
        # Información clínica y de seguridad
        "indications_and_usage": get_first_value(record, "indications_and_usage"),
        "contraindications": get_first_value(record, "contraindications"),
        "warnings": get_first_value(record, "warnings"),
        "precautions": get_first_value(record, "precautions"),
        "adverse_reactions": get_first_value(record, "adverse_reactions"),
        "drug_interactions": get_first_value(record, "drug_interactions"),
        "dosage_and_administration": get_first_value(record, "dosage_and_administration"),
        "clinical_studies": get_first_value(record, "clinical_studies"),
        
        # Identificadores y metadatos
        "set_id": record.get("set_id"),
        "id": record.get("id"),
        "effective_time": record.get("effective_time")
    }
    
    return transformed

In [14]:
transformed_record = transform_record(first_record)

for field, value in transformed_record.items():
    if value is not None:
        print(f"{field}: {str(value)[:150]}")

clinical_pharmacology: CLINICAL PHARMACOLOGY Pharmacokinetics: Serum, urine and tear concentrations of ofloxacin were measured in 30 healthy women at various time points dur
pharmacokinetics: Pharmacokinetics: Serum, urine and tear concentrations of ofloxacin were measured in 30 healthy women at various time points during a ten-day course o
indications_and_usage: INDICATIONS AND USAGE Ofloxacin ophthalmic solution is indicated for the treatment of infections caused by susceptible strains of the following bacter
contraindications: CONTRAINDICATIONS Ofloxacin ophthalmic solution is contraindicated in patients with a history of hypersensitivity to ofloxacin, to other quinolones, o
warnings: WARNINGS NOT FOR INJECTION. Ofloxacin ophthalmic solution should not be injected subconjunctivally, nor should it be introduced directly into the ante
precautions: PRECAUTIONS General: As with other anti-infectives, prolonged use may result in overgrowth of nonsusceptible organisms, including fungi. If

In [15]:
transformed_data = [
    transform_record(record)
    for record in data["results"]
]

print(f"Registros transformados: {len(transformed_data)}")
print(f"Variables por registro: {len(transformed_data[0])}")

Registros transformados: 5
Variables por registro: 24


## 9. Extracción de datos

Una vez validada la conexión con la API y definida la estructura de los registros, se realizará la extracción del dataset de trabajo.

Se seleccionarán registros que contengan información tanto de `pharmacokinetics` como de `pharmacodynamics`. Se utilizará paginación mediante los parámetros `limit` y `skip`, ya que la API limita la cantidad de registros recuperables en cada petición.

Para esta etapa se establece un objetivo inicial de 15.000 registros. La extracción se realizará mediante un script reproducible y con manejo de errores para evitar la pérdida o corrupción de los datos ante posibles fallos de conexión o de la API.

### Resultado de la extracción

La extracción fue ejecutada mediante el script `src/extract_data.py` desde el entorno `openfda_ds`.

El proceso finalizó correctamente y produjo:

- **15.000 registros extraídos**
- **24 variables por registro**
- Archivo generado: `data/data_extracted.json`

El dataset completo se conserva localmente para evitar cargar innecesariamente un archivo de gran tamaño en memoria durante esta etapa. Para el repositorio se incluirá una muestra representativa de los datos.

In [17]:
import json
import ijson

sample_records = []

with open("data/data_extracted.json", "rb") as file:
    for record in ijson.items(file, "item"):
        sample_records.append(record)

        if len(sample_records) >= 50:
            break

with open("data/sample_data.json", "w", encoding="utf-8") as file:
    json.dump(
        sample_records,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Muestra creada correctamente.")
print("Registros:", len(sample_records))
print("Variables por registro:", len(sample_records[0]))

Muestra creada correctamente.
Registros: 50
Variables por registro: 24
